# PyTorch to TensorFlow Lite Converter

Model: SimpleCNN for Banana Leaf Disease Classification
Classes: 0 = Black Sigatoka, 1 = Fusarium Wilt, 2 = Healthy

**Upload your banana_model.pth file to Google Colab before running!**

In [ ]:
# Install required packages (usually pre-installed on Colab)
!pip install onnx onnx-tf -q

## Step 1: Define Your Model Architecture (SimpleCNN)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.fc1 = nn.Linear(32 * 30 * 30, 128)
        self.fc2 = nn.Linear(128, 3)  # 3 classes

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## Step 2: Load Your Model

Upload your banana_model.pth file to Colab (folder icon on the left)

In [ ]:
# Load the model
model = SimpleCNN()
checkpoint = torch.load('banana_model.pth', map_location='cpu')

# Handle different checkpoint formats
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
elif 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
else:
    model.load_state_dict(checkpoint)

model.eval()
print("Model loaded successfully!")

## Step 3: Convert to ONNX

Note: Your model uses 128x128 input size

In [ ]:
# Create dummy input (batch_size=1, channels=3, height=128, width=128)
dummy_input = torch.randn(1, 3, 128, 128)

# Export to ONNX
onnx_path = 'banana_leaf_model.onnx'
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)
print(f"ONNX model saved to: {onnx_path}")

## Step 4: Convert ONNX to TensorFlow

In [ ]:
import onnx
from onnx_tf.backend import prepare

# Load ONNX model
onnx_model = onnx.load(onnx_path)

# Prepare for TensorFlow
tf_rep = prepare(onnx_model)

# Save TensorFlow model
tf_saved_model_dir = 'tf_saved_model'
tf_rep.export_graph(tf_saved_model_dir)
print(f"TensorFlow SavedModel saved to: {tf_saved_model_dir}")

## Step 5: Convert TensorFlow to TensorFlow Lite

In [ ]:
import tensorflow as tf

# Create TFLite converter
converter = tf.lite.TFLiteConverter.from_saved_model(tf_saved_model_dir)

# Set optimization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Set supported ops
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

# Convert
tflite_model = converter.convert()

# Save
tflite_path = 'leaf_disease_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"TensorFlow Lite model saved to: {tflite_path}")

# Get file size
import os
size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

## Step 6: Download the TFLite Model

In [ ]:
# Download the model
from google.colab import files
files.download(tflite_path)

print("Download started! Save the file.")

---

## After Downloading

1. Copy `leaf_disease_model.tflite` to:
   ```
   bananaapp/assets/leaf_disease_model.tflite
   ```

2. Run in your Flutter project:
   ```bash
   flutter pub get
   flutter build apk --release
   ```

## Class Labels
- Index 0: Black Sigatoka
- Index 1: Fusarium Wilt
- Index 2: Healthy